In [1]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings("ignore")

In [18]:
DATA_PATH = "../data/interim/deliveries_loaded.csv"
df = pd.read_csv(DATA_PATH)

In [21]:
df.shape

(43739, 16)

In [22]:
df.head()

,Order_ID,Agent_Age,Agent_Rating,Store_Latitude,Store_Longitude,Drop_Latitude,Drop_Longitude,Order_Date,Order_Time,Pickup_Time,Weather,Traffic,Vehicle,Area,Delivery_Time,Category
0,ialx566343618,37,4.9,22.745049,75.892471,22.765049,75.912471,2022-03-19,11:30:00,11:45:00,Sunny,High,motorcycle,Urban,120,Clothing
1,akqg208421122,34,4.5,12.913041,77.683237,13.043041,77.813237,2022-03-25,19:45:00,19:50:00,Stormy,Jam,scooter,Metropolitian,165,Electronics
2,njpu434582536,23,4.4,12.914264,77.678400,12.924264,77.688400,2022-03-19,08:30:00,08:45:00,Sandstorms,Low,motorcycle,Urban,130,Sports
3,rjto796129700,38,4.7,11.003669,76.976494,11.053669,77.026494,2022-04-05,18:00:00,18:10:00,Sunny,Medium,motorcycle,Metropolitian,105,Cosmetics
4,zguw716275638,32,4.6,12.972793,80.249982,13.012793,80.289982,2022-03-26,13:30:00,13:45:00,Cloudy,High,scooter,Metropolitian,150,Toys


In [23]:
df.duplicated().sum()

np.int64(0)

In [27]:
df["Order_Date"] = pd.to_datetime(df["Order_Date"], errors="coerce")
df["Order_Time"] = pd.to_datetime(
    df["Order_Time"], format="%H:%M:%S", errors="coerce"
).dt.time

df["Pickup_Time"] = pd.to_datetime(
    df["Pickup_Time"], format="%H:%M:%S", errors="coerce"
).dt.time

In [28]:
df.head()

,Order_ID,Agent_Age,Agent_Rating,Store_Latitude,Store_Longitude,Drop_Latitude,Drop_Longitude,Order_Date,Order_Time,Pickup_Time,Weather,Traffic,Vehicle,Area,Delivery_Time,Category
0,ialx566343618,37,4.9,22.745049,75.892471,22.765049,75.912471,2022-03-19,11:30:00,11:45:00,Sunny,High,motorcycle,Urban,120,Clothing
1,akqg208421122,34,4.5,12.913041,77.683237,13.043041,77.813237,2022-03-25,19:45:00,19:50:00,Stormy,Jam,scooter,Metropolitian,165,Electronics
2,njpu434582536,23,4.4,12.914264,77.678400,12.924264,77.688400,2022-03-19,08:30:00,08:45:00,Sandstorms,Low,motorcycle,Urban,130,Sports
3,rjto796129700,38,4.7,11.003669,76.976494,11.053669,77.026494,2022-04-05,18:00:00,18:10:00,Sunny,Medium,motorcycle,Metropolitian,105,Cosmetics
4,zguw716275638,32,4.6,12.972793,80.249982,13.012793,80.289982,2022-03-26,13:30:00,13:45:00,Cloudy,High,scooter,Metropolitian,150,Toys


In [29]:
df[["Order_Time", "Pickup_Time"]].isna().sum()

Order_Time     91
Pickup_Time     0
dtype: int64

In [30]:
df = df.dropna(subset=["Order_Time", "Pickup_Time"])

In [32]:
df["Order_DateTime"] = pd.to_datetime(
    df["Order_Date"].astype(str) + " " + df["Order_Time"].astype(str),
    errors="coerce"
)

df["Pickup_DateTime"] = pd.to_datetime(
    df["Order_Date"].astype(str) + " " + df["Pickup_Time"].astype(str),
    errors="coerce"
)

In [33]:
df["Pickup_Next_Day"] = (df["Pickup_Time"] < df["Order_Time"]).astype(int)
df.loc[df["Pickup_Next_Day"] == 1, "Pickup_DateTime"] += pd.Timedelta(days=1)

In [34]:
(df["Pickup_DateTime"] < df["Order_DateTime"]).sum()

np.int64(0)

In [35]:
df.isna().sum()

Order_ID            0
Agent_Age           0
Agent_Rating       54
Store_Latitude      0
Store_Longitude     0
Drop_Latitude       0
Drop_Longitude      0
Order_Date          0
Order_Time          0
Pickup_Time         0
Weather             0
Traffic             0
Vehicle             0
Area                0
Delivery_Time       0
Category            0
Order_DateTime      0
Pickup_DateTime     0
Pickup_Next_Day     0
dtype: int64

In [36]:
num_cols = ["Agent_Rating"]

for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

In [37]:
df.isna().sum()

Order_ID           0
Agent_Age          0
Agent_Rating       0
Store_Latitude     0
Store_Longitude    0
Drop_Latitude      0
Drop_Longitude     0
Order_Date         0
Order_Time         0
Pickup_Time        0
Weather            0
Traffic            0
Vehicle            0
Area               0
Delivery_Time      0
Category           0
Order_DateTime     0
Pickup_DateTime    0
Pickup_Next_Day    0
dtype: int64

In [38]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 43648 entries, 0 to 43738
Data columns (total 19 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   Order_ID         43648 non-null  object        
 1   Agent_Age        43648 non-null  int64         
 2   Agent_Rating     43648 non-null  float64       
 3   Store_Latitude   43648 non-null  float64       
 4   Store_Longitude  43648 non-null  float64       
 5   Drop_Latitude    43648 non-null  float64       
 6   Drop_Longitude   43648 non-null  float64       
 7   Order_Date       43648 non-null  datetime64[ns]
 8   Order_Time       43648 non-null  object        
 9   Pickup_Time      43648 non-null  object        
 10  Weather          43648 non-null  object        
 11  Traffic          43648 non-null  object        
 12  Vehicle          43648 non-null  object        
 13  Area             43648 non-null  object        
 14  Delivery_Time    43648 non-null  int64     

In [42]:
R = 6371  # Earth radius in km

def deg_to_rad(degrees):
    return degrees * (np.pi / 180)

def distcalculate(lat1, lon1, lat2, lon2):
    d_lat = deg_to_rad(lat2 - lat1)
    d_lon = deg_to_rad(lon2 - lon1)
    a = (
        np.sin(d_lat / 2) ** 2
        + np.cos(deg_to_rad(lat1))
        * np.cos(deg_to_rad(lat2))
        * np.sin(d_lon / 2) ** 2
    )
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    return R * c

df["Distance"] = np.nan

for i in range(len(df)):
    df.iloc[i, df.columns.get_loc("Distance")] = distcalculate(
        df.iloc[i]["Store_Latitude"],
        df.iloc[i]["Store_Longitude"],
        df.iloc[i]["Drop_Latitude"],
        df.iloc[i]["Drop_Longitude"],
    )

In [43]:
df.head()

,Order_ID,Agent_Age,Agent_Rating,Store_Latitude,Store_Longitude,Drop_Latitude,Drop_Longitude,Order_Date,Order_Time,Pickup_Time,Weather,Traffic,Vehicle,Area,Delivery_Time,Category,Order_DateTime,Pickup_DateTime,Pickup_Next_Day,Distance
0,ialx566343618,37,4.9,22.745049,75.892471,22.765049,75.912471,2022-03-19,11:30:00,11:45:00,Sunny,High,motorcycle,Urban,120,Clothing,2022-03-19 11:30:00,2022-03-19 11:45:00,0,3.025149
1,akqg208421122,34,4.5,12.913041,77.683237,13.043041,77.813237,2022-03-25,19:45:00,19:50:00,Stormy,Jam,scooter,Metropolitian,165,Electronics,2022-03-25 19:45:00,2022-03-25 19:50:00,0,20.183530
2,njpu434582536,23,4.4,12.914264,77.678400,12.924264,77.688400,2022-03-19,08:30:00,08:45:00,Sandstorms,Low,motorcycle,Urban,130,Sports,2022-03-19 08:30:00,2022-03-19 08:45:00,0,1.552758
3,rjto796129700,38,4.7,11.003669,76.976494,11.053669,77.026494,2022-04-05,18:00:00,18:10:00,Sunny,Medium,motorcycle,Metropolitian,105,Cosmetics,2022-04-05 18:00:00,2022-04-05 18:10:00,0,7.790401
4,zguw716275638,32,4.6,12.972793,80.249982,13.012793,80.289982,2022-03-26,13:30:00,13:45:00,Cloudy,High,scooter,Metropolitian,150,Toys,2022-03-26 13:30:00,2022-03-26 13:45:00,0,6.210138


In [44]:
df["Distance"].describe()

count    43648.000000
mean        27.255432
std        303.815765
min          1.465067
25%          4.663432
50%          9.220419
75%         13.682165
max       6884.726399
Name: Distance, dtype: float64

In [50]:
df["Extreme_Distance_Flag"] = (df["Distance"] > 100).astype(int)
df["Extreme_Distance_Flag"].value_counts()

Extreme_Distance_Flag
0    43492
1      156
Name: count, dtype: int64

In [51]:
df["Pickup_Next_Day"] = (df["Pickup_Time"] < df["Order_Time"]).astype(int)
df["Pickup_Next_Day"].value_counts()

Pickup_Next_Day
0    42820
1      828
Name: count, dtype: int64

In [54]:
hours = df["Order_Time"].apply(lambda t: t.hour)
df["Night_Order_Flag"] = ((hours >= 22) | (hours <= 5)).astype(int)
df["Night_Order_Flag"].value_counts()

Night_Order_Flag
0    34163
1     9485
Name: count, dtype: int64

In [58]:
df.head()

,Order_ID,Agent_Age,Agent_Rating,Store_Latitude,Store_Longitude,Drop_Latitude,Drop_Longitude,Order_Date,Order_Time,Pickup_Time,...,Vehicle,Area,Delivery_Time,Category,Order_DateTime,Pickup_DateTime,Pickup_Next_Day,Distance,Extreme_Distance_Flag,Night_Order_Flag
0,ialx566343618,37,4.9,22.745049,75.892471,22.765049,75.912471,2022-03-19,11:30:00,11:45:00,...,motorcycle,Urban,120,Clothing,2022-03-19 11:30:00,2022-03-19 11:45:00,0,3.025149,0,0
1,akqg208421122,34,4.5,12.913041,77.683237,13.043041,77.813237,2022-03-25,19:45:00,19:50:00,...,scooter,Metropolitian,165,Electronics,2022-03-25 19:45:00,2022-03-25 19:50:00,0,20.183530,0,0
2,njpu434582536,23,4.4,12.914264,77.678400,12.924264,77.688400,2022-03-19,08:30:00,08:45:00,...,motorcycle,Urban,130,Sports,2022-03-19 08:30:00,2022-03-19 08:45:00,0,1.552758,0,0
3,rjto796129700,38,4.7,11.003669,76.976494,11.053669,77.026494,2022-04-05,18:00:00,18:10:00,...,motorcycle,Metropolitian,105,Cosmetics,2022-04-05 18:00:00,2022-04-05 18:10:00,0,7.790401,0,0
4,zguw716275638,32,4.6,12.972793,80.249982,13.012793,80.289982,2022-03-26,13:30:00,13:45:00,...,scooter,Metropolitian,150,Toys,2022-03-26 13:30:00,2022-03-26 13:45:00,0,6.210138,0,0


In [59]:
df["Weather"].value_counts(normalize=True)

Weather
Fog           0.170455
Stormy        0.168942
Cloudy        0.166972
Sandstorms    0.165987
Windy         0.165483
Sunny         0.162161
Name: proportion, dtype: float64

No weather is "rare"

In [60]:
df["Vehicle"].value_counts(normalize=True)

Vehicle
motorcycle     0.584655
scooter        0.334655
van            0.080691
Name: proportion, dtype: float64

In [61]:
df["Area"].value_counts(normalize=True)

Area
Metropolitian     0.747663
Urban             0.222828
Other             0.026026
Semi-Urban        0.003482
Name: proportion, dtype: float64

In [62]:
rare_area = df["Area"].value_counts(normalize=True)
rare_area = rare_area[rare_area < 0.01].index

df["Area"] = df["Area"].replace(rare_area, "other")

In [63]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 43648 entries, 0 to 43738
Data columns (total 22 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   Order_ID               43648 non-null  object        
 1   Agent_Age              43648 non-null  int64         
 2   Agent_Rating           43648 non-null  float64       
 3   Store_Latitude         43648 non-null  float64       
 4   Store_Longitude        43648 non-null  float64       
 5   Drop_Latitude          43648 non-null  float64       
 6   Drop_Longitude         43648 non-null  float64       
 7   Order_Date             43648 non-null  datetime64[ns]
 8   Order_Time             43648 non-null  object        
 9   Pickup_Time            43648 non-null  object        
 10  Weather                43648 non-null  object        
 11  Traffic                43648 non-null  object        
 12  Vehicle                43648 non-null  object        
 13  Area  

In [64]:
df.head()

,Order_ID,Agent_Age,Agent_Rating,Store_Latitude,Store_Longitude,Drop_Latitude,Drop_Longitude,Order_Date,Order_Time,Pickup_Time,...,Vehicle,Area,Delivery_Time,Category,Order_DateTime,Pickup_DateTime,Pickup_Next_Day,Distance,Extreme_Distance_Flag,Night_Order_Flag
0,ialx566343618,37,4.9,22.745049,75.892471,22.765049,75.912471,2022-03-19,11:30:00,11:45:00,...,motorcycle,Urban,120,Clothing,2022-03-19 11:30:00,2022-03-19 11:45:00,0,3.025149,0,0
1,akqg208421122,34,4.5,12.913041,77.683237,13.043041,77.813237,2022-03-25,19:45:00,19:50:00,...,scooter,Metropolitian,165,Electronics,2022-03-25 19:45:00,2022-03-25 19:50:00,0,20.183530,0,0
2,njpu434582536,23,4.4,12.914264,77.678400,12.924264,77.688400,2022-03-19,08:30:00,08:45:00,...,motorcycle,Urban,130,Sports,2022-03-19 08:30:00,2022-03-19 08:45:00,0,1.552758,0,0
3,rjto796129700,38,4.7,11.003669,76.976494,11.053669,77.026494,2022-04-05,18:00:00,18:10:00,...,motorcycle,Metropolitian,105,Cosmetics,2022-04-05 18:00:00,2022-04-05 18:10:00,0,7.790401,0,0
4,zguw716275638,32,4.6,12.972793,80.249982,13.012793,80.289982,2022-03-26,13:30:00,13:45:00,...,scooter,Metropolitian,150,Toys,2022-03-26 13:30:00,2022-03-26 13:45:00,0,6.210138,0,0


In [65]:
df.to_csv(
    "../data/interim/deliveries_cleaned.csv",index=False
)